# Learning Objectives
In this notebook, you will learn Spark Dataframe APIs.

# Question List

Solve the following questions using Spark Dataframe APIs

### Join

1. easy - https://pgexercises.com/questions/joins/simplejoin.html
2. easy - https://pgexercises.com/questions/joins/simplejoin2.html
3. easy - https://pgexercises.com/questions/joins/self2.html 
4. medium - https://pgexercises.com/questions/joins/threejoin.html (three join)
5. medium - https://pgexercises.com/questions/joins/sub.html (subquery and join)

### Aggregation

1. easy - https://pgexercises.com/questions/aggregates/count3.html Group by order by
2. easy - https://pgexercises.com/questions/aggregates/fachours.html group by order by
3. easy - https://pgexercises.com/questions/aggregates/fachoursbymonth.html group by with condition 
4. easy - https://pgexercises.com/questions/aggregates/fachoursbymonth2.html group by multi col
5. easy - https://pgexercises.com/questions/aggregates/members1.html count distinct
6. med - https://pgexercises.com/questions/aggregates/nbooking.html group by multiple cols, join

### String & Date

1. easy - https://pgexercises.com/questions/string/concat.html format string
2. easy - https://pgexercises.com/questions/string/case.html WHERE + string function
3. easy - https://pgexercises.com/questions/string/reg.html WHERE + string function
4. easy - https://pgexercises.com/questions/string/substr.html group by, substr
5. easy - https://pgexercises.com/questions/date/series.html generate ts
6. easy - https://pgexercises.com/questions/date/bookingspermonth.html extract month from ts

In [0]:
base_path = "/Volumes/test_flight_data/default/pgexercises/"

files = {
    "bookings": "bookings.csv",
    "members": "members.csv",
    "facilities": "facilities.csv"
}

for table_name, file_name in files.items():
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(base_path + file_name)
    )
    
    df.createOrReplaceTempView(table_name)

### Question

How can you produce a list of the start times for bookings by members named 'David Farrell'?

https://pgexercises.com/questions/joins/simplejoin.html

In [0]:
df_david = spark.sql("SELECT starttime from bookings JOIN members ON bookings.memid = members.memid WHERE surname = 'Farrell' AND firstname = 'David'")
display(df_david)

starttime
2012-09-18T09:00:00.000Z
2012-09-18T17:30:00.000Z
2012-09-18T13:30:00.000Z
2012-09-18T20:00:00.000Z
2012-09-19T09:30:00.000Z
2012-09-19T15:00:00.000Z
2012-09-19T12:00:00.000Z
2012-09-20T15:30:00.000Z
2012-09-20T11:30:00.000Z
2012-09-20T14:00:00.000Z


### Question

In [0]:
df_2 = spark.sql("SELECT starttime, name FROM bookings JOIN facilities ON bookings.facid = facilities.facid WHERE name LIKE 'Tennis%' AND starttime >= '2012-09-21' AND starttime < '2012-09-22' ORDER BY starttime ASC;")

display(df_2)

starttime,name
2012-09-21T08:00:00.000Z,Tennis Court 2
2012-09-21T08:00:00.000Z,Tennis Court 1
2012-09-21T09:30:00.000Z,Tennis Court 1
2012-09-21T10:00:00.000Z,Tennis Court 2
2012-09-21T11:30:00.000Z,Tennis Court 2
2012-09-21T12:00:00.000Z,Tennis Court 1
2012-09-21T13:30:00.000Z,Tennis Court 1
2012-09-21T14:00:00.000Z,Tennis Court 2
2012-09-21T15:30:00.000Z,Tennis Court 1
2012-09-21T16:00:00.000Z,Tennis Court 2


### Question

In [0]:
df_3 = spark.sql("select mems.firstname as memfname, mems.surname as memsname, recs.firstname as recfname, recs.surname as recsname from members mems left outer join members recs on recs.memid = mems.recommendedby ORDER BY memsname, memfname;")

display(df_3)

memfname,memsname,recfname,recsname
Florence,Bader,Ponder,Stibbons
Anne,Baker,Ponder,Stibbons
Timothy,Baker,Jemima,Farrell
Tim,Boothe,Tim,Rownam
Gerald,Butters,Darren,Smith
Joan,Coplin,Timothy,Baker
Erica,Crumpet,Tracy,Smith
Nancy,Dare,Janice,Joplette
David,Farrell,null,null
Jemima,Farrell,null,null


### Question

In [0]:
df_4 = spark.sql("SELECT DISTINCT CONCAT(m.firstname, ' ', m.surname) AS member, f.name AS facility FROM bookings b JOIN members m ON b.memid = m.memid JOIN facilities f ON b.facid = f.facid WHERE f.name LIKE 'Tennis Court%' ORDER BY member, facility;")

display(df_4)

member,facility
Anne Baker,Tennis Court 1
Anne Baker,Tennis Court 2
Burton Tracy,Tennis Court 1
Burton Tracy,Tennis Court 2
Charles Owen,Tennis Court 1
Charles Owen,Tennis Court 2
Darren Smith,Tennis Court 2
David Farrell,Tennis Court 1
David Farrell,Tennis Court 2
David Jones,Tennis Court 1


### Question

In [0]:
df_5 = spark.sql("""
SELECT DISTINCT
    CONCAT(m.firstname, ' ', m.surname) AS member,
    (
      SELECT MAX(CONCAT(r.firstname, ' ', r.surname))
      FROM members r
      WHERE r.memid = m.recommendedby
    ) AS recommender
FROM members m
ORDER BY member
""")

display(df_5)

member,recommender
Anna Mackenzie,Darren Smith
Anne Baker,Ponder Stibbons
Burton Tracy,null
Charles Owen,Darren Smith
Darren Smith,null
David Farrell,null
David Jones,Janice Joplette
David Pinker,Jemima Farrell
Douglas Jones,David Jones
Erica Crumpet,Tracy Smith


### Question

In [0]:
df_6 = spark.sql("""
SELECT recommendedby, COUNT(recommendedby) AS count FROM members 
WHERE recommendedby IS NOT NULL 
GROUP BY recommendedby 
ORDER BY recommendedby ASC;
""")

display(df_6.limit(5))

recommendedby,count
1,5
2,3
3,1
4,2
5,1


### Question 

In [0]:
df_7 = spark.sql("""
SELECT facid, SUM(slots) as `Total Slots` FROM bookings 
GROUP BY facid
ORDER BY facid;
""")

display(df_7)

facid,Total Slots
0,1320
1,1278
2,1209
3,830
4,1404
5,228
6,1104
7,908
8,911


### Question

In [0]:
df_8 = spark.sql("""
SELECT facid, SUM(slots) as `Total Slots` FROM bookings 
WHERE starttime >= '2012-09-01' AND starttime < '2012-10-01'
GROUP BY facid
ORDER BY `Total Slots`;
""")

display(df_8)

facid,Total Slots
5,122
3,422
7,426
8,471
6,540
2,570
1,588
0,591
4,648


### Question

In [0]:
df_9 = spark.sql("""
SELECT facid, EXTRACT(MONTH FROM starttime) AS month, SUM(slots) as `Total Slots` FROM bookings 
WHERE starttime >= '2012-01-01' AND starttime < '2013-01-01'
GROUP BY facid, month
ORDER BY facid,month;
""")

display(df_9)

facid,month,Total Slots
0,7,270
0,8,459
0,9,591
1,7,207
1,8,483
1,9,588
2,7,180
2,8,459
2,9,570
3,7,104


### Question


### Question


In [0]:
df_10 = spark.sql("""
select count(distinct memid) from bookings;
""")

display(df_10)

count(DISTINCTmemid)
30


### Question


In [0]:
df_11 = spark.sql("""
WITH ranked_bookings AS (
    SELECT
        m.surname,
        m.firstname,
        m.memid,
        b.starttime,
        ROW_NUMBER() OVER (
            PARTITION BY m.memid
            ORDER BY b.starttime
        ) AS rn
    FROM members m
    JOIN bookings b
        ON m.memid = b.memid
    WHERE b.starttime >= '2012-09-01'
)
SELECT
    surname,
    firstname,
    memid,
    starttime
FROM ranked_bookings
WHERE rn = 1
ORDER BY memid, starttime
""")

display(df_11)

surname,firstname,memid,starttime
GUEST,GUEST,0,2012-09-01T08:00:00.000Z
Smith,Darren,1,2012-09-01T09:00:00.000Z
Smith,Tracy,2,2012-09-01T11:30:00.000Z
Rownam,Tim,3,2012-09-01T16:00:00.000Z
Joplette,Janice,4,2012-09-01T15:00:00.000Z
Butters,Gerald,5,2012-09-02T12:30:00.000Z
Tracy,Burton,6,2012-09-01T15:00:00.000Z
Dare,Nancy,7,2012-09-01T12:30:00.000Z
Boothe,Tim,8,2012-09-01T08:30:00.000Z
Stibbons,Ponder,9,2012-09-01T11:00:00.000Z


### Question

In [0]:
df_12 = spark.sql("SELECT CONCAT(surname,', ',firstname) FROM members;")
display(df_12)


"CONCAT(surname,', ',firstname)"
"GUEST, GUEST"
"Smith, Darren"
"Smith, Tracy"
"Rownam, Tim"
"Joplette, Janice"
"Butters, Gerald"
"Tracy, Burton"
"Dare, Nancy"
"Boothe, Tim"
"Stibbons, Ponder"


### Question

In [0]:
df_13 = spark.sql("""
SELECT *
FROM facilities
WHERE LOWER(name) LIKE 'tennis%';
""")

display(df_13)

facid,name,membercost,guestcost,initialoutlay,monthlymaintenance
0,Tennis Court 1,5.0,25.0,10000,200
1,Tennis Court 2,5.0,25.0,8000,200


### Question

In [0]:
df_14 = spark.sql("""
SELECT memid, telephone FROM members WHERE telephone LIKE '%(%)%';
""")

display(df_14)

memid,telephone
0,(000) 000-0000
3,(844) 693-0723
4,(833) 942-4710
5,(844) 078-4130
6,(822) 354-9973
7,(833) 776-4001
8,(811) 433-2547
9,(833) 160-3900
10,(855) 542-5251
11,(844) 536-8036


### Question

In [0]:
df_15 = spark.sql("""
SELECT SUBSTR(surname,1,1) as letter, count(*) as count 
FROM members
GROUP BY letter 
ORDER BY letter;
""")

display(df_15)

letter,count
B,5
C,2
D,1
F,2
G,2
H,1
J,3
M,1
O,1
P,2


### Question

In [0]:
df_16 = spark.sql("select explode(sequence(to_date('2012-10-01'), to_date('2012-10-31'), interval 1 day)) as ts")
display(df_16) 

ts
2012-10-01
2012-10-02
2012-10-03
2012-10-04
2012-10-05
2012-10-06
2012-10-07
2012-10-08
2012-10-09
2012-10-10


### Question

In [0]:
df_17 = spark.sql("""
select date_trunc('month', starttime) as month, count(*)
from bookings
group by month
order by month  
""")
display(df_17) 

month,count(*)
2012-07-01T00:00:00.000Z,658
2012-08-01T00:00:00.000Z,1472
2012-09-01T00:00:00.000Z,1913
2013-01-01T00:00:00.000Z,1
